![WEkEO by Copernicus and Partners](https://github.com/wekeo/frameworks/blob/main/img/WEkEO_banner_EUMETSAT.png?raw=true)

<div style="display: flex; justify-content: space-between;">
  <a href="./README.md">&lt;&lt; README</a>
  <a href="./02_xcubeViewer_JH_visualize_remote_datacubes.ipynb">Visualizing remote data cubes &gt;&gt;</a>
</div>

Copyright (c) 2023 European Union 

License: MIT

# Create and visualize a data cube using WEkEO and the xcube Viewer

**This notebook can only be run on the WEkEO JupyterHub.**

[![launch-WEKEO](https://img.shields.io/badge/launch-WEKEO-1a4696.svg)](https://jupyterhub.prod.wekeo2.eu/hub/user-redirect/lab/tree/public/wekeo4data/wekeo-xcube/01_xcubeViewer_JH_create_and_visualize_data_cube.ipynb)
[![github](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/wekeo/wekeo-xcube/blob/main/01_xcubeViewer_JH_create_and_visualize_data_cube.ipynb)

**PREREQUISITES**

This notebook has the following prerequisites:
- [A WEkEO account](https://my.wekeo.eu/user-registration) if you are using or plan to use WEkEO.
- access and execution of this notebook inside the [WEkEO JupyterHub](https://jupyterhub.prod.wekeo2.eu/)
- some knowledge about the data access through the [Earthkit WEkEO Plugins](https://github.com/wekeo/wekeo4data/blob/main/wekeo-earthkit/01_WEkEOEarthkit_Introduction.ipynb)

**LEARNING OUTCOMES**

At the end of this notebook you will know:

- how to create xcube-compatible data cubes from WEkEO data 
- how to correctly write a config file for xcube Viewer
- how to visualize and interact with data in the xcube viewe

**DATA USED**

| Product Description  | WEkEO HDA ID | WEkEO metadata |
|:--------------------:|:-------------:|:-----------------:|
| ERA5 hourly data on single levels from 1940 to present  | EO:ECMWF:DAT:REANALYSIS_ERA5_SINGLE_LEVELS | [link](https://www.wekeo.eu/data?view=dataset&dataset=EO%3AECMWF%3ADAT%3AREANALYSIS_ERA5_SINGLE_LEVELS) |

**CONTENTS**

1. [Import](#1-import)  
2. [Download datasets using the Earthkit plugin](#2-download-datasets-using-the-earthkit-plugin)  
3. [Creating a datacube with xarray](#3-creating-a-datacube-with-xarray)  
4. [Creating a config file](#4-creating-a-config-file)  
5. [Viewing the data with xcube Viewer](#5-viewing-the-data-with-xcube-viewer)  
6. [Take home messages](#6-take-home-messages)

<hr>

The [xcube](https://xcube.readthedocs.io/en/latest/index.html) software package has been developed to generate, manipulate, analyse, and publish data cubes from Earth Observation data. This notebook shows how xcube, especially the xcube viewer can be used with EO data coming from WEkEO.

## 1. Import

In [1]:

import os                                       # a library that allows us access to basic operating system commands like making directories    
import earthkit.data as ekd                     # a library for seamless data access and conversion to xarray
from earthkit.data import settings, cache       # modules helpful for configuring the earthkit package
import xarray as xr                             # a library that supports the use of multi-dimensional arrays in Python
from xcube.webapi.viewer import Viewer          # a library to visualize datacubes inside the Jupyter Notebook 
import getpass
import hda
from pathlib import Path

Set the xcube Viewer url to show the xcube Viewer inside the notebook if you are running this notebook on the WEkEO Jupyter Hub.

In [2]:
if "WEKEO_DATABROKER_URL" in os.environ:
    os.environ["XCUBE_JUPYTER_LAB_URL"] = f"https://jupyterhub.prod.wekeo2.eu/user/{os.environ['JUPYTERHUB_USER']}/"

### Setup the WEkEO HDA and credentials

To use the WEkEO HDA adaptor, create a file called `.hdarc` in your home directory:

Add the following lines to the file:

`user:<your_user_name>`

`password:<your_password>`

For more information about the HDA and how to set up your credentials, check out [this notebook](https://github.com/wekeo/wekeo-hda/blob/master/wekeo_harmonised_data_access_api.ipynb).

The Earthkit WEkEO plugin reads your WEkEO credentials from the same `.hdarc` file.

Lets create our credentials file, if it doesn't already exist.

In [ ]:
 # load credentials
wekeo_credentials_file = Path(Path.home() / '.hdarc' )

def get_credentials():
    user = input('Enter your WEkEO user name: ').strip()
    password = getpass.getpass('Enter your WEkEO password: ').strip()
    return user, password

def save_credentials(user, password):
    os.makedirs(os.path.dirname(wekeo_credentials_file), exist_ok=True)
    with open(wekeo_credentials_file, "w", encoding="utf-8") as f:
        f.write(f'user: {user}\npassword: {password}')

# Try loading saved credentials
if not wekeo_credentials_file.is_file():
    user, password = get_credentials()
    save_credentials(user, password)

# Try to get token, retry if credentials are invalid
while True:
    try:
        token = hda.Client().token
        print(f"Your WEkEO token: '{token}'")
        break  # Exit loop if successful
    except hda.api.ConfigurationError as e:
        print("Invalid username or password. Please try again.")
        print (e)
        user, password = get_credentials()
        save_credentials(user, password)

## 2. Download datasets using the Earthkit plugin

In this notebook we will work with the **ERA5 Air temperature reanalysis**. You can get more information on this dataset and explore it <a href="https://www.wekeo.eu/data?view=dataset&dataset=EO%3AECMWF%3ADAT%3AREANALYSIS_ERA5_SINGLE_LEVELS" target="_blank">here</a>. 

The API request is derived from the WEkEO Viewer. 

We will download and transform the data to xarray using the Earthkit WEkEO Plugin.  

In [3]:
s = {"cache-policy": "user",
     "user-cache-directory": "./cache"}
settings.set(s)
cache.directory()

'./cache'

In [4]:
era5_request =ekd.from_source("wekeo", 
  "EO:ECMWF:DAT:REANALYSIS_ERA5_SINGLE_LEVELS_MONTHLY_MEANS",
  request = {
  "dataset_id": "EO:ECMWF:DAT:REANALYSIS_ERA5_SINGLE_LEVELS_MONTHLY_MEANS",
  "product_type": ["monthly_averaged_reanalysis_by_hour_of_day"],
  "variable": ["2m_temperature"],
  "year": ["2019"],
  "month": ["01"],
  "time": ["00:00","01:00","02:00","03:00","04:00","05:00",
            "06:00","07:00","08:00","09:00","10:00","11:00",
            "12:00","13:00","14:00","15:00","16:00","17:00",
            "18:00","19:00","20:00","21:00","22:00","23:00"],
  "data_format": "netcdf",
  "download_format": "zip",
  "itemsPerPage": 200,
  "startIndex": 0
})

In [5]:
# download dataset and convert it to xarray

era5_ds = era5_request.to_xarray()

In [6]:
era5_ds

<xarray.Dataset> Size: 100MB
Dimensions:     (valid_time: 24, latitude: 721, longitude: 1440)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 192B 2019-01-01 ... 2019-01-01T23...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    expver      (valid_time) <U4 384B dask.array<chunksize=(24,), meta=np.ndarray>
Data variables:
    t2m         (valid_time, latitude, longitude) float32 100MB dask.array<chunksize=(12, 361, 720), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-09-29T13:28 GRIB to CDM+CF via cfgrib-0.9.1...

## 3. Creating a datacube with xarray

From the xarray overview we see, that the longitude is in the range of `[0, 360]`. For compatibility reasons we convert it to the range of `[-180, 180]` and remane the time variable to `time`. 

In [7]:
era5_ds.coords['longitude'] = (era5_ds.coords['longitude'] + 180) % 360 - 180
era5_ds = era5_ds.rename({"valid_time": "time"})
era5_ds = era5_ds.drop_vars(["number", "expver"])
era5_ds_reshaped = era5_ds.sortby(era5_ds.longitude)



Subset the global dataset to our area of interest os Spain:

In [10]:
spain_era5 = era5_ds_reshaped.sel( latitude=slice(44.1, 35.6), longitude = slice(-10, 4))[['t2m']]
spain_era5.attrs["title"] = "Air Temp Spain 2019-01-01"

In [11]:
spain_era5

<xarray.Dataset> Size: 187kB
Dimensions:    (time: 24, latitude: 34, longitude: 57)
Coordinates:
  * time       (time) datetime64[ns] 192B 2019-01-01 ... 2019-01-01T23:00:00
  * latitude   (latitude) float64 272B 44.0 43.75 43.5 ... 36.25 36.0 35.75
  * longitude  (longitude) float64 456B -10.0 -9.75 -9.5 -9.25 ... 3.5 3.75 4.0
Data variables:
    t2m        (time, latitude, longitude) float32 186kB dask.array<chunksize=(12, 34, 40), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-09-29T13:28 GRIB to CDM+CF via cfgrib-0.9.1...
    title:                   Air Temp Spain 2019-01-01

## 4. Creating a config file

To add data cubes to the xcube viewer, they have to be defined in a config file.

Two main parts of the config file is the ``Datasets`` description and the ``Styles`` description. As our datacube is already in our environment, we don't have to explicitly define the ``Datasets`` parameters it in the config file. So we are only setting the ``Styles`` parameters. 

The ``Style`` description contains:
* **Identifier** *unique name for the style*
* **ColorMappings** 
    * **VariableName** *must be identical to the variable name in the cube*
        * **ColorBar** *name of colorbar as in matplotlib*
        * **ValueRange** *range of the dataset*

In [ ]:
viewer = Viewer(server_config={
    "Styles": [
        {
            "Identifier": "temperature",
            "ColorMappings": {
                "t2m": {
                    "ValueRange": [250.0, 300.0],
                    "ColorBar": "inferno"
                },
            }
        }
    ]
})

Next, we add our dataset to the Viewer

In [10]:
viewer.add_dataset(spain_era5, style="temperature")

'5d44a471-d768-4800-96a9-d22f8de1ac9d'

## 5. Viewing the data with xcube Viewer

The `viewer.show()` function shows an instance of the xcube Viewer inside your norebook. 

You can now explore the datacubes by <font color="#138D75"> browse through time, create a time series at one point, compare different locations ...</font>

In [ ]:
viewer.show()

<img src='./img/01_xcubeviewer_show.png' align='left' width='80%'/>

## 6. Take home messages

- Data from WEkEO can be turned into an xcube-compatible data cube with xarray, e.g. by converting the longitudes to `[-180, 180]`.
- The xcube Viewer is configured with a server configuration that contains the `Styles` (and optionally the `Datasets`).
- `viewer.show()` opens the xcube Viewer inside the notebook, so you can browse through time and compare locations.

<hr>

[View on GitHub](https://github.com/wekeo) | 
[Visit WEkEO](https://wekeo.copernicus.eu) | [Run Notebooks on the WEkEO Workspace](https://jupyterhub.prod.wekeo2.eu/) | 
[Contact helpdesk for support](mailto:support@wekeo.eu)